In [1]:
import asyncio
import math
import pandas as pd

from itertools import chain, repeat

In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [ ]:
from ib_insync import *
from ibkr.Class_IBKR_IB import IBKR_IB
ibkr = IBKR_IB(port=7496)

async def start_ibkr():
    await ibkr.connect()
    print("IBKR connected:", ibkr.ib.isConnected())

await start_ibkr()

IBKR connected: True


Error 200, reqId 27: No security definition has been found for the request, contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=1.0, right='C', exchange='SMART', currency='USD')
Error 200, reqId 28: No security definition has been found for the request, contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=1.0, right='P', exchange='SMART', currency='USD')
Error 200, reqId 29: No security definition has been found for the request, contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=2.0, right='C', exchange='SMART', currency='USD')
Error 200, reqId 31: No security definition has been found for the request, contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=3.0, right='C', exchange='SMART', currency='USD')
Error 200, reqId 30: No security definition has been found for the request, contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=2.0, right='P', exch

In [4]:
async def make_chains_from_symbols(symbols_list, product_type):

    contracts     = []
    details       = []
    option_chains = []

    for symbol in symbols_list:

        if product_type == 'equity':
            contract = Stock(symbol=symbol, exchange='SMART', currency="USD")
            fut_exch = ""
        elif product_type == "future":
            contract = Future(localSymbol=symbol, exchange='CME', currency="USD")
            fut_exch = "CME"

        contract = await ibkr.ib.qualifyContractsAsync(contract) # this may lead to a printed line since its return has nowhere to be mapped
        contracts.append(contract[0])

        print(contract[0])

        detail = await ibkr.ib.reqContractDetailsAsync(contract[0])
        details.append(detail[0])

        print(detail[0])

        option_chain = await ibkr.ib.reqSecDefOptParamsAsync(underlyingSymbol=detail[0].contract.symbol,
                                                             futFopExchange=fut_exch,
                                                             underlyingSecType=detail[0].contract.secType,
                                                             underlyingConId=detail[0].contract.conId
                                                            )
        option_chains.append(option_chain)

        print(option_chain[0])

        # print('count=', len(stocl_option_chain[0].expirations), stock_option_chain[0].expirations)
        # print('count=', len(stock_option_chain[0].strikes), stock_option_chain[0].strikes)
        # print("2 *", len(stock_option_chain[0].expirations), "*", len(stock_option_chain[0].strikes), "=", 
        #                       len(stock_option_chain[0].expirations) * len(stock_option_chain[0].strikes) * 2)

        # print('\n')

    return option_chains, details, contracts

In [5]:
stock_symbols = ['IBIT']

s_chain, s_detail, s_contracts = await make_chains_from_symbols(stock_symbols, 'equity')


future_symbols = [
                  'BTCM6',
                  'BTCN6',
                  'BTCQ6',
                  'BTCU6',
                  'BTCV6',
                  'BTCX6',
                  'BTCZ6'
                  ]


f_chain, f_detail, f_contracts = await make_chains_from_symbols(future_symbols, 'future')

combined_zip = chain(zip(s_chain, s_detail, repeat("equity")), zip(f_chain, f_detail, repeat("future")))

Stock(conId=677037673, symbol='IBIT', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='IBIT', tradingClass='NMS')
ContractDetails(contract=Contract(secType='STK', conId=677037673, symbol='IBIT', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='IBIT', tradingClass='NMS'), marketName='NMS', minTick=0.01, orderTypes='ACTIVETIM,AD,ADDONT,ADJUST,ALERT,ALGO,ALLOC,AON,AVGCOST,BASKET,BENCHPX,CASHQTY,COND,CONDORDER,DARKONLY,DARKPOLL,DAY,DEACT,DEACTDIS,DEACTEOD,DIS,DUR,GAT,GTC,GTD,GTT,HID,IBKRATS,ICE,IMB,IOC,LIT,LMT,LOC,MIDPX,MIT,MKT,MOC,MTL,NGCOMB,NODARK,NONALGO,OCA,OPG,OPGREROUT,PEGBENCH,PEGMID,POSTATS,POSTONLY,PREOPGRTH,PRICECHK,REL,REL2MID,RELPCTOFS,RPI,RTH,SCALE,SCALEODD,SCALERST,SIZECHK,SNAPMID,SNAPMKT,SNAPREL,STP,STPLMT,SWEEP,TRAIL,TRAILLIT,TRAILLMT,TRAILMIT,WHATIF', validExchanges='SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,ISLAND,DRCTEDGE,BEX,BATS,EDGEA,BYX,IEX,EDGX,FOXRIVER,PEARL,NYSENAT,LTSE,MEMX,IBEOS,OVERNIGHT,TPLUS0,PSX,T24X', priceMagnif

In [6]:
option_contracts = []

for chain, details, product_type in combined_zip:
    for expiry in chain[0].expirations:
        for strike in chain[0].strikes:
            for right in ['C', 'P']:
                
                if product_type == 'equity':
                    option_contract = Option(lastTradeDateOrContractMonth=expiry,
                                                    strike=float(strike),
                                                    right=right,
                                                    symbol=details.contract.symbol,
                                                    exchange=details.contract.exchange,
                                                    currency=details.contract.currency,
                                                    )
                
                elif product_type == 'future':
                    option_contract = FuturesOption(lastTradeDateOrContractMonth=expiry,
                                                    strike=float(strike),
                                                    right=right,
                                                    symbol=details.contract.symbol,
                                                    exchange=details.contract.exchange,
                                                    currency=details.contract.currency,
                                                    )
                option_contracts.append(option_contract)

option_contracts = await ibkr.ib.qualifyContractsAsync(*option_contracts) # this may lead to printed lines since its return has nowhere to be mapped

# print(len(option_contracts))

Unknown contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=1.0, right='C', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=1.0, right='P', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=2.0, right='C', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=2.0, right='P', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=3.0, right='C', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=3.0, right='P', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=4.0, right='C', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='I

In [7]:
def clean_num(x):
    """
    Convert IBKR nan / -1 / None style missing values to None.
    """
    if x is None:
        return None
    try:
        if math.isnan(x):
            return None
    except TypeError:
        pass
    if x == -1:
        return None
    return x

In [8]:
def ticker_row(ticker):
    contract = ticker.contract
    return {
            "conId": getattr(ticker.contract, "conId", None),
            "secType": getattr(contract, "secType", None),
            "symbol": getattr(contract, "symbol", None),
            "expiration": getattr(contract, "lastTradeDateOrContractMonth", None),
            "right": getattr(contract, "right", None),
            "strike": getattr(contract, "strike", None),

            "close": clean_num(getattr(ticker, "close", None)),
        #    "volume": clean_num(getattr(ticker, "volume", None)),
            "avgVolume": clean_num(getattr(ticker, "avVolume", None)),
            "futuresOpenInterest": clean_num(getattr(ticker, "futuresOpenInterest", None)),
            "putOpenInterest": clean_num(getattr(ticker, "putOpenInterest", None)),
            "callOpenInterest": clean_num(getattr(ticker, "callOpenInterest", None)),
        }

In [9]:
async def get_data(contracts, batch_size=50, wait_seconds=30):
    
    generic_ticks = "100,101,165,588"
    all_rows = []

    for i in range(0, len(contracts), batch_size):
        batch = contracts[i:i + batch_size]
        tickers = []

        print(f"Requesting {i} to {i + len(batch) - 1} of {len(contracts)}")

        for contract in batch:
            ticker = ibkr.ib.reqMktData(
                contract,
                genericTickList=generic_ticks,
                snapshot=False,
            )
            tickers.append(ticker)

        await asyncio.sleep(wait_seconds)

        rows = [ticker_row(ticker) for ticker in tickers]
        all_rows.extend(rows)

        # IMPORTANT: cancel streaming data before next batch
        for ticker in tickers:
            ibkr.ib.cancelMktData(ticker.contract)

        await asyncio.sleep(2)  # small pause between batches

    return pd.DataFrame(all_rows)

In [10]:
linear_contracts = [*f_contracts, *s_contracts] 
df_linear  = await get_data(linear_contracts, batch_size = 25, wait_seconds=60)
df_linear


Requesting 0 to 7 of 8


,conId,secType,symbol,expiration,right,strike,close,avgVolume,futuresOpenInterest,putOpenInterest,callOpenInterest
0,751356962,FUT,BRR,20260626,,0.0,62945.000000,NaN,0.0,0.0,0.0
1,850790355,FUT,BRR,20260731,,0.0,63240.000000,NaN,0.0,0.0,0.0
2,859040542,FUT,BRR,20260828,,0.0,63495.000000,NaN,0.0,0.0,0.0
3,772435574,FUT,BRR,20260925,,0.0,63745.000000,NaN,0.0,0.0,0.0
4,876880607,FUT,BRR,20261030,,0.0,64115.000000,NaN,0.0,0.0,0.0
5,887699043,FUT,BRR,20261127,,0.0,64425.000000,NaN,0.0,0.0,0.0
6,751356958,FUT,BRR,20261224,,0.0,64680.000000,NaN,0.0,0.0,0.0
7,677037673,STK,IBIT,,,0.0,36.360001,440638.0,NaN,0.0,0.0


In [11]:
df_options = await get_data(option_contracts, batch_size = 25, wait_seconds=60)
df_options

Requesting 0 to 24 of 6714
Requesting 25 to 49 of 6714
Requesting 50 to 74 of 6714
Requesting 75 to 99 of 6714
Requesting 100 to 124 of 6714
Requesting 125 to 149 of 6714
Requesting 150 to 174 of 6714
Requesting 175 to 199 of 6714
Requesting 200 to 224 of 6714
Requesting 225 to 249 of 6714
Requesting 250 to 274 of 6714
Requesting 275 to 299 of 6714
Requesting 300 to 324 of 6714
Requesting 325 to 349 of 6714
Requesting 350 to 374 of 6714
Requesting 375 to 399 of 6714
Requesting 400 to 424 of 6714
Requesting 425 to 449 of 6714
Requesting 450 to 474 of 6714
Requesting 475 to 499 of 6714
Requesting 500 to 524 of 6714
Requesting 525 to 549 of 6714
Requesting 550 to 574 of 6714
Requesting 575 to 599 of 6714
Requesting 600 to 624 of 6714
Requesting 625 to 649 of 6714
Requesting 650 to 674 of 6714
Requesting 675 to 699 of 6714
Requesting 700 to 724 of 6714
Requesting 725 to 749 of 6714
Requesting 750 to 774 of 6714
Requesting 775 to 799 of 6714
Requesting 800 to 824 of 6714
Requesting 825 to 8

,conId,secType,symbol,expiration,right,strike,close,avgVolume,futuresOpenInterest,putOpenInterest,callOpenInterest
0,890245019,OPT,IBIT,20260622,C,19.0,17.39,None,None,0.0,0.0
1,890246741,OPT,IBIT,20260622,P,19.0,0.00,None,None,0.0,0.0
2,890245072,OPT,IBIT,20260622,C,20.0,16.39,None,None,0.0,0.0
3,890246786,OPT,IBIT,20260622,P,20.0,0.00,None,None,0.0,0.0
4,890245132,OPT,IBIT,20260622,C,21.0,15.39,None,None,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
6709,756512288,FOP,BRR,20261224,P,500000.0,426905.00,None,None,0.0,0.0
6710,756512281,FOP,BRR,20261224,C,550000.0,5.00,None,None,0.0,0.0
6711,756512251,FOP,BRR,20261224,P,550000.0,475935.00,None,None,0.0,0.0
6712,756512303,FOP,BRR,20261224,C,600000.0,0.00,None,None,0.0,0.0


In [12]:
df_option_conIds = (
    df_options.pivot_table(
        index=["symbol", "strike"],
        columns=["expiration", "right"],
        values="conId",
        aggfunc="first"
    )
    .sort_index()
    .sort_index(axis=1)
    .reset_index()
)

df_option_conIds


expiration symbol   strike 20260622     20260624         20260626  \
right                             C   P        C   P            C   
0             BRR   1000.0      NaN NaN      NaN NaN  751506503.0   
1             BRR   5000.0      NaN NaN      NaN NaN  751506548.0   
2             BRR  10000.0      NaN NaN      NaN NaN  751506561.0   
3             BRR  17500.0      NaN NaN      NaN NaN  852657225.0   
4             BRR  20000.0      NaN NaN      NaN NaN  764688659.0   
..            ...      ...      ...  ..      ...  ..          ...   
527          IBIT    120.0      NaN NaN      NaN NaN          NaN   
528          IBIT    125.0      NaN NaN      NaN NaN          NaN   
529          IBIT    130.0      NaN NaN      NaN NaN          NaN   
530          IBIT    135.0      NaN NaN      NaN NaN          NaN   
531          IBIT    140.0      NaN NaN      NaN NaN          NaN   

expiration              20260629      ...     20270617               \
right                 P        C   P  ...            C            P   
0           751506545.0      NaN NaN  ...          NaN          NaN   
1           751506518.0      NaN NaN  ...          NaN          NaN   
2           751506558.0      NaN NaN  ...          NaN          NaN   
3           852657015.0      NaN NaN  ...          NaN          NaN   
4           764688665.0      NaN NaN  ...          NaN          NaN   
..                  ...      ...  ..  ...          ...          ...   
527                 NaN      NaN NaN  ...  812339736.0  812340411.0   
528                 NaN      NaN NaN  ...  812339784.0  812340474.0   
529                 NaN      NaN NaN  ...  820670481.0  820670611.0   
530                 NaN      NaN NaN  ...  820670532.0  820670659.0   
531                 NaN      NaN NaN  ...  820670569.0  820670702.0   

expiration     20271217                  20280121              20280616      \
right                 C            P            C            P        C   P   
0                   NaN          NaN          NaN          NaN      NaN NaN   
1                   NaN          NaN          NaN          NaN      NaN NaN   
2                   NaN          NaN          NaN          NaN      NaN NaN   
3                   NaN          NaN          NaN          NaN      NaN NaN   
4                   NaN          NaN          NaN          NaN      NaN NaN   
..                  ...          ...          ...          ...      ...  ..   
527         796581180.0  796581256.0  816970623.0  816970877.0      NaN NaN   
528         799997845.0  799998059.0  816970678.0  816970947.0      NaN NaN   
529         799997915.0  799998137.0  820670860.0  820670974.0      NaN NaN   
530         799997984.0  799998214.0  820670896.0  820671018.0      NaN NaN   
531         820670736.0  820670814.0  820670934.0  820671049.0      NaN NaN   

expiration 20281215      
right             C   P  
0               NaN NaN  
1               NaN NaN  
2               NaN NaN  
3               NaN NaN  
4               NaN NaN  
..              ...  ..  
527             NaN NaN  
528             NaN NaN  
529             NaN NaN  
530             NaN NaN  
531             NaN NaN  

[532 rows x 66 columns]

In [ ]:
df_option_prices = (
    df_options.pivot_table(
        index=["symbol", "strike"],
        columns=["expiration", "right"],
        values="close",
        aggfunc="first"
    )
    .sort_index()
    #.sort_index(axis=1)
    .reset_index()
)

df_option_prices

expiration symbol   strike 20260622     20260624     20260626      20260629  \
right                             C   P        C   P        C    P        C   
0             BRR   1000.0      NaN NaN      NaN NaN  61895.0  0.0      NaN   
1             BRR   5000.0      NaN NaN      NaN NaN  57900.0  0.0      NaN   
2             BRR  10000.0      NaN NaN      NaN NaN  52905.0  0.0      NaN   
3             BRR  17500.0      NaN NaN      NaN NaN  45410.0  0.0      NaN   
4             BRR  20000.0      NaN NaN      NaN NaN  42910.0  0.0      NaN   
..            ...      ...      ...  ..      ...  ..      ...  ...      ...   
527          IBIT    120.0      NaN NaN      NaN NaN      NaN  NaN      NaN   
528          IBIT    125.0      NaN NaN      NaN NaN      NaN  NaN      NaN   
529          IBIT    130.0      NaN NaN      NaN NaN      NaN  NaN      NaN   
530          IBIT    135.0      NaN NaN      NaN NaN      NaN  NaN      NaN   
531          IBIT    140.0      NaN NaN      NaN NaN      NaN  NaN      NaN   

expiration      ... 20270617         20271217         20280121          \
right        P  ...        C       P        C       P        C       P   
0          NaN  ...      NaN     NaN      NaN     NaN      NaN     NaN   
1          NaN  ...      NaN     NaN      NaN     NaN      NaN     NaN   
2          NaN  ...      NaN     NaN      NaN     NaN      NaN     NaN   
3          NaN  ...      NaN     NaN      NaN     NaN      NaN     NaN   
4          NaN  ...      NaN     NaN      NaN     NaN      NaN     NaN   
..          ..  ...      ...     ...      ...     ...      ...     ...   
527        NaN  ...     0.30   83.64     0.73   83.64     0.83   83.64   
528        NaN  ...     0.27   88.64     0.67   88.64     0.77   88.64   
529        NaN  ...     0.25   93.64     0.62   93.64     0.70   93.64   
530        NaN  ...     0.23   98.64     0.57   98.64     0.65   98.64   
531        NaN  ...     0.22  103.64     0.55  103.64     0.61  103.64   

expiration 20280616     20281215      
right             C   P        C   P  
0               NaN NaN      NaN NaN  
1               NaN NaN      NaN NaN  
2               NaN NaN      NaN NaN  
3               NaN NaN      NaN NaN  
4               NaN NaN      NaN NaN  
..              ...  ..      ...  ..  
527             NaN NaN      NaN NaN  
528             NaN NaN      NaN NaN  
529             NaN NaN      NaN NaN  
530             NaN NaN      NaN NaN  
531             NaN NaN      NaN NaN  

[532 rows x 66 columns]

In [16]:
df_option_prices.to_csv("file.csv")

In [ ]:
df_option_prices.columns = pd.MultiIndex.from_tuples([
    ("strike", "K") if col == ("strike", "") or col == "strike"
    else (col, "") if not isinstance(col, tuple)
    else col
    for col in df_option_prices.columns
])

[stock_close] = df_linear.loc[df_linear['secType'] == 'STK', 'close'] 
df_option_prices.insert(
    loc=1,                 # between symbol and strike
    column=("stock", "S"),
    value=stock_close         # your constant
)

In [19]:
for expiration in df_option_prices.columns.get_level_values(0).unique():

    call_col = (expiration, "C")
    put_col  = (expiration, "P")

    cps_col = (expiration, "C-P-S")
    kcps_col = (expiration, "K-(C-P-S)")

    rate_col  = (expiration, "combo_return_rate")

    tv_col = (expiration, "time_value")


    if call_col in df_option_prices.columns and put_col in df_option_prices.columns:

        df_option_prices[cps_col] = (
            df_option_prices[call_col] - df_option_prices[put_col] - df_option_prices[("stock", "S")])

        df_option_prices[kcps_col] = (
            df_option_prices[("strike", "K")] + df_option_prices[cps_col])
    
        df_option_prices[rate_col] = (
            df_option_prices[kcps_col] / -df_option_prices[cps_col])

        df_option_prices[tv_col] = df_option_prices[[call_col, put_col]].min(axis=1)
    
df_option_prices.sort_index(axis=1)
df_option_prices

symbol      stock   strike 20260622     20260624     20260626       \
                    S        K        C   P        C   P        C    P   
0      BRR  36.360001   1000.0      NaN NaN      NaN NaN  61895.0  0.0   
1      BRR  36.360001   5000.0      NaN NaN      NaN NaN  57900.0  0.0   
2      BRR  36.360001  10000.0      NaN NaN      NaN NaN  52905.0  0.0   
3      BRR  36.360001  17500.0      NaN NaN      NaN NaN  45410.0  0.0   
4      BRR  36.360001  20000.0      NaN NaN      NaN NaN  42910.0  0.0   
..     ...        ...      ...      ...  ..      ...  ..      ...  ...   
527   IBIT  36.360001    120.0      NaN NaN      NaN NaN      NaN  NaN   
528   IBIT  36.360001    125.0      NaN NaN      NaN NaN      NaN  NaN   
529   IBIT  36.360001    130.0      NaN NaN      NaN NaN      NaN  NaN   
530   IBIT  36.360001    135.0      NaN NaN      NaN NaN      NaN  NaN   
531   IBIT  36.360001    140.0      NaN NaN      NaN NaN      NaN  NaN   

    20260629  ...          20280121            20280616            \
           C  ... combo_return_rate time_value    C-P-S K-(C-P-S)   
0        NaN  ...               NaN        NaN      NaN       NaN   
1        NaN  ...               NaN        NaN      NaN       NaN   
2        NaN  ...               NaN        NaN      NaN       NaN   
3        NaN  ...               NaN        NaN      NaN       NaN   
4        NaN  ...               NaN        NaN      NaN       NaN   
..       ...  ...               ...        ...      ...       ...   
527      NaN  ...          0.006965       0.83      NaN       NaN   
528      NaN  ...          0.006198       0.77      NaN       NaN   
529      NaN  ...          0.005414       0.70      NaN       NaN   
530      NaN  ...          0.004838       0.65      NaN       NaN   
531      NaN  ...          0.004376       0.61      NaN       NaN   

                                 20281215                              \
    combo_return_rate time_value    C-P-S K-(C-P-S) combo_return_rate   
0                 NaN        NaN      NaN       NaN               NaN   
1                 NaN        NaN      NaN       NaN               NaN   
2                 NaN        NaN      NaN       NaN               NaN   
3                 NaN        NaN      NaN       NaN               NaN   
4                 NaN        NaN      NaN       NaN               NaN   
..                ...        ...      ...       ...               ...   
527               NaN        NaN      NaN       NaN               NaN   
528               NaN        NaN      NaN       NaN               NaN   
529               NaN        NaN      NaN       NaN               NaN   
530               NaN        NaN      NaN       NaN               NaN   
531               NaN        NaN      NaN       NaN               NaN   

                
    time_value  
0          NaN  
1          NaN  
2          NaN  
3          NaN  
4          NaN  
..         ...  
527        NaN  
528        NaN  
529        NaN  
530        NaN  
531        NaN  

[532 rows x 195 columns]

df_option_prices = df_option_prices.sort_index(axis=1)
df_option_prices.to_csv("file.csv")